# NucleiSky3D: API Workflow Example

This notebook demonstrates a complete 3D workflow using **NucleiSky3D**: load data, optionally create a synthetic demo crop with a known similarity transform, match using pyramid/hash matchers, generate QC MIPs, and export the aligned crop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from nucleisky3d.demo_utils import generate_random_subvolume_3d
from nucleisky3d.io import get_voxel_size_um_from_tiff, load_volume
from nucleisky3d.pipeline import NucleiSky3D
from nucleisky3d.segmentation import segment_nuclei_2p5d
from nucleisky3d.visualization import plot_warp_overlay3D
from nucleisky3d.export import export_aligned_crop_tiff

from nucleisky3d.features import extract_nuclear_features_3d, centroids_from_df_3d


In [ ]:
# --- 1. Configuration ---

# File paths (adjust to your data)
base_path = Path("path/to/your/data")
full_volume_path = base_path / "full_volume.tif"
crop_volume_path = base_path / "crop_volume.tif"  # optional if using demo crop
output_dir = Path("nucleisky3d_output")
output_dir.mkdir(exist_ok=True)

# Demo crop settings (used when crop_volume_path does not exist)
use_demo_crop_if_missing = True
crop_shape_zyx = (64, 128, 128)
scale_range = (0.7, 1.4)
random_seed = 17

# Optional: set to 2 or 4 for very large volumes to reduce MIP memory usage
mip_downsample = 1


In [ ]:
# --- 2. Load data (or generate synthetic full volume) ---

def _synthetic_volume(seed: int = 0, shape=(96, 160, 160), n_blobs: int = 60):
    rng = np.random.default_rng(seed)
    vol = np.zeros(shape, dtype=np.float32)
    zz, yy, xx = np.meshgrid(
        np.arange(shape[0]), np.arange(shape[1]), np.arange(shape[2]), indexing="ij"
    )
    for _ in range(n_blobs):
        z0 = rng.integers(0, shape[0])
        y0 = rng.integers(0, shape[1])
        x0 = rng.integers(0, shape[2])
        sigma = rng.uniform(1.5, 3.0)
        amp = rng.uniform(0.6, 1.2)
        dist2 = (zz - z0) ** 2 + (yy - y0) ** 2 + (xx - x0) ** 2
        vol += amp * np.exp(-dist2 / (2 * sigma**2))
    vol = vol / (vol.max() + 1e-6)
    return vol.astype(np.float32)

if full_volume_path.exists():
    img_full = load_volume(str(full_volume_path))
    voxel_size_full_um = get_voxel_size_um_from_tiff(str(full_volume_path))
else:
    print("Full volume not found; using synthetic data.")
    img_full = _synthetic_volume(seed=random_seed)
    voxel_size_full_um = (1.0, 1.0, 1.0)
# WARNING: this isotropic tuple is a placeholder fallback for the demo only.
# Replace with acquisition-validated voxel metadata in (Z, Y, X) order.

print("Full volume shape:", img_full.shape)
print("Full voxel size (µm):", voxel_size_full_um)


In [ ]:
# --- 3. Load crop or create a synthetic demo crop with known transform ---

rng = np.random.default_rng(random_seed)

if crop_volume_path.exists():
    img_crop = load_volume(str(crop_volume_path))
    voxel_size_crop_um = get_voxel_size_um_from_tiff(str(crop_volume_path))
    ground_truth = None
else:
    if not use_demo_crop_if_missing:
        raise FileNotFoundError("Crop volume not found. Set use_demo_crop_if_missing=True to synthesize.")
    img_crop, voxel_size_crop_um, ground_truth = generate_random_subvolume_3d(
        img_full,
        crop_shape_zyx=crop_shape_zyx,
        scale_range=scale_range,
        voxel_size_um=voxel_size_full_um,
        rng=rng,
    )

print("Crop volume shape:", img_crop.shape)
print("Crop voxel size (µm):", voxel_size_crop_um)
if ground_truth:
    print("Ground truth scale:", ground_truth["scale"])


In [ ]:
# --- 4. Segment nuclei in 3D ---

seg_kwargs = dict(
    threshold_method="otsu",
    gaussian_sigma=1.0,
    min_object_size=50,
    do_watershed=True,
)

labels_full = segment_nuclei_2p5d(img_full, method="threshold", pixel_size_um_zyx=tuple(voxel_size_full_um), settings={"threshold": seg_kwargs})
labels_crop = segment_nuclei_2p5d(img_crop, method="threshold", pixel_size_um_zyx=tuple(voxel_size_crop_um), settings={"threshold": seg_kwargs})

print("Full labels:", labels_full.max(), "Crop labels:", labels_crop.max())

# Precompute 3D features once (can be reused by all matcher calls)
df_full = extract_nuclear_features_3d(labels_full, pixel_size_um=tuple(voxel_size_full_um))
df_crop = extract_nuclear_features_3d(labels_crop, pixel_size_um=tuple(voxel_size_crop_um))
print(df_full[["label", "centroid_z_px", "centroid_y_px", "centroid_x_px"]].head())

centroids_full_um = centroids_from_df_3d(df_full, use_um=True)
centroids_crop_um = centroids_from_df_3d(df_crop, use_um=True)
print("Centroids (full, crop):", centroids_full_um.shape, centroids_crop_um.shape)


In [ ]:
# --- 5. Run matching with pyramid + hashing ---

result_pyramid = NucleiSky3D(
    centroids_crop_um=centroids_crop_um,
    centroids_full_um=centroids_full_um,
    full_shape_px_zyx=img_full.shape,
    crop_shape_px_zyx=img_crop.shape,
    pixel_size_full_um_zyx=voxel_size_full_um,
    pixel_size_crop_um_zyx=voxel_size_crop_um,
    matcher="pyramid",
    df_full=df_full,
    df_crop=df_crop,
)

result_hash = NucleiSky3D(
    centroids_crop_um=centroids_crop_um,
    centroids_full_um=centroids_full_um,
    full_shape_px_zyx=img_full.shape,
    crop_shape_px_zyx=img_crop.shape,
    pixel_size_full_um_zyx=voxel_size_full_um,
    pixel_size_crop_um_zyx=voxel_size_crop_um,
    matcher="hashing",
    df_full=df_full,
    df_crop=df_crop,
)

print("Pyramid success:", result_pyramid["success"], "Scale:", result_pyramid["best_scale"])
print("Hash success:", result_hash["success"], "Scale:", result_hash["best_scale"])

if ground_truth:
    print("GT scale:", ground_truth["scale"])


In [ ]:
# --- 6. QC overlay + export (use the pyramid result if successful) ---

best_result = result_pyramid if result_pyramid["success"] else result_hash
if not best_result["success"]:
    raise RuntimeError("Matching failed. Try adjusting segmentation settings or scale_range.")

plot_warp_overlay3D(
    img_full_zyx=img_full,
    img_crop_zyx=img_crop,
    record_or_result=best_result,
    pixel_size_full_um_zyx=voxel_size_full_um,
    pixel_size_crop_um_zyx=voxel_size_crop_um,
    save_path=output_dir / "qc_overlay_3d.png",
    show=False,
)

export_path = export_aligned_crop_tiff(
    img_full,
    img_crop,
    output_path=output_dir / "aligned_crop.tif",
    pixel_size_full_um=voxel_size_full_um,
    pixel_size_crop_um=voxel_size_crop_um,
    best_scale=best_result["best_scale"],
    best_R=best_result["best_R"],
    best_t=best_result["best_t"],
)

print("Aligned crop exported to:", export_path)
